In [1]:
import mediapy
import numpy as np
from tqdm import tqdm

from robokit.datasets.tcl_datasets import TCLDataset
from robokit.debug_utils import print_batch
from robokit.debug_utils import concatenate_rgb_images, plot_action_wrt_time, plot_force_sensor_wrt_time


In [4]:
# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/1201_debug/")
# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/1117_red_peper_pick_place/")
# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/1117_carrot_pick_place/")
# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/1024_wipe_white_board/")
# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/1024_pour_water/")
# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/1024_eggs_pick_place/")
# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/1024_sweep_bean/")
# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/1021_sweep_bean/")
# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/1021_temp/")
# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/1201_wipe_blackboard/")
# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/1201_pour_water/")
# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/1201_screw_bulb/")
# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/1201_screw_bulb_turn_off/")
# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/1201_screw_bulb_turn_off_object/")
# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/1201_screw_bulb_turn_off_light/")
# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/0209_tower_boby_hard/")
# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/0209_tower_boby_easy/")
# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/0417_put_mouse/")
# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/0417_ethernet/")
# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/0417_brush/")
# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/260514_ethernet/")
# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/tmp/")
# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/1014_lh_air_fryer/")
dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/1029_lh_chemistry/")


# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/0417_plastic_egg/")
# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/0417_put_mouse_hard/")
# dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/1014_lh_tower_three_blocks/")
print("Total tasks:", len(dataset.tasks))
print_batch('TCLDataset', dataset[0])

[TCLDataset] total length: 691
Total tasks: 1
TCLDataset: Dict, keys=['primary_rgb', 'gripper_rgb', 'primary_depth', 'gripper_depth', 'language_text', 'actions', 'rel_actions', 'robot_obs', 'force_torque', 'gripper_max_effort']
--primary_rgb, <class 'numpy.ndarray'>, shape=(480, 848, 3), min=0.0000, max=255.0000, dtype=uint8
--gripper_rgb, <class 'numpy.ndarray'>, shape=(480, 848, 3), min=0.0000, max=255.0000, dtype=uint8
--primary_depth, <class 'numpy.ndarray'>, shape=(480, 848), min=0.0000, max=14680.0000, dtype=uint16
--gripper_depth, <class 'numpy.ndarray'>, shape=(480, 848), min=0.0000, max=65535.0000, dtype=uint16
--language_text: <class 'str'>, len=558, value='Perform an acid-base neutralization task: First, press down to toggle the power switch on and pick up the dropper.Next, enter a titration loop—squeeze and slightly release the gripper to draw alkaline solution from the reservoir, lift the dropper over the reaction beaker, squeeze to dispense the liquid, and return the drop

In [5]:
global_idx = 0

watch_idx = task_skip_idx = -1  # -1:last
if watch_idx < 0:
    watch_idx = task_skip_idx = len(dataset.tasks) + watch_idx
vis_max_frames = 3500
fps = 30
sample_rate = 10  # 采样率，每sample_rate帧取一帧，以减少内存使用。例如，3表示每3帧取1帧。

# task_skip_idx = 35
# fps = 5

print("Total tasks:", len(dataset.tasks))

task_instruction = None
for task_idx, task in enumerate(dataset.tasks):
    task_length = min(vis_max_frames, dataset.task_lengths[task_idx])
    effective_length = (task_length + sample_rate - 1) // sample_rate  # 计算有效帧数，确保覆盖整个长度

    # 只存储必要的列表，移除未使用的images_primary和images_gripper
    images_cat = []
    actions = []
    has_gripper_effort = 'gripper_max_effort' in dataset.load_keys
    gripper_efforts = []
    force_torque = []

    if task_idx < task_skip_idx:
        global_idx += dataset.task_lengths[task_idx]
        continue

    for i in tqdm(range(effective_length)):
        # 计算实际索引，实现间隔抽帧
        actual_frame_idx = i * sample_rate
        if actual_frame_idx >= task_length:
            break  # 防止越界

        frame_data = dataset[global_idx + actual_frame_idx]

        images_cat.append(concatenate_rgb_images(frame_data['primary_rgb'], frame_data['gripper_rgb'], vertical=True))
        actions.append(frame_data['rel_actions'])
        if has_gripper_effort:
            gripper_efforts.append(float(frame_data['gripper_max_effort'][0]))
        force_torque.append(frame_data['force_torque'])

        task_instruction = frame_data['language_text']

    # 在循环后更新global_idx到下一个task
    global_idx += dataset.task_lengths[task_idx]

    all_vis = images_cat
    
    # Vis force and action
    # Use the same sampled frames as the camera and action curves.
    gripper_max_effort_T = (
        np.asarray(gripper_efforts, dtype=np.float32) if has_gripper_effort else None
    )
    actions_vis, fig, ax = plot_action_wrt_time(
        np.array(actions), gripper_max_effort_T=gripper_max_effort_T,
    )
    force_torque_vis, _, _ = plot_force_sensor_wrt_time(np.array(force_torque))
    all_vis = []
    for frame_idx in range(len(images_cat)):  # 使用effective_length的长度
        # 先拼接相机图像和动作可视化
        temp_vis = concatenate_rgb_images(
            images_cat[frame_idx],
            force_torque_vis[frame_idx],
            vertical=False,
            resize_ratio=1
        )
        # 再拼接力传感器可视化
        all_vis.append(concatenate_rgb_images(
            temp_vis,
            actions_vis[frame_idx],
            vertical=False,
            resize_ratio=1
        ))

    print("Instruction:", task_instruction)
    # mediapy.show_video(all_vis, fps=fps / sample_rate)  # 可选：调整fps以匹配采样率，保持视频时长类似
    mediapy.show_video(all_vis, fps=fps)  # 可选：speed up

    break

Total tasks: 1


100%|██████████████████████████████████| 70/70 [00:00<00:00, 78.06it/s]


Plotting action dynamic figures...
Plotting force sensor dynamic figures...
Instruction: Perform an acid-base neutralization task: First, press down to toggle the power switch on and pick up the dropper.Next, enter a titration loop—squeeze and slightly release the gripper to draw alkaline solution from the reservoir, lift the dropper over the reaction beaker, squeeze to dispense the liquid, and return the dropper to the storage beaker—repeatingthis cycle until the solution in the reaction beaker turns red. Once neutralized, release the dropper, return to the homeconfiguration, close the gripper, and press down to turn off the power switch.


This browser does not support the video tag.


In [ ]:
dataset = TCLDataset(root="/home/geyuan/local_soft/TCL/1017_sweep_bean")
global_idx = 0

task_skip_idx = 0
fps = 30

print("Total tasks:", len(dataset.tasks))

for task_idx, task in enumerate(dataset.tasks):
    task_length = dataset.task_lengths[task_idx]
    images_primary, images_gripper = [], []
    images_cat = []
    actions = []
    has_gripper_effort = 'gripper_max_effort' in dataset.load_keys
    gripper_efforts = []
    print("task_idx = ",task_idx)

    if task_idx < task_skip_idx:
        global_idx += task_length
        continue

    for frame_idx in tqdm(range(task_length)):
        frame_data = dataset[global_idx]
        global_idx += 1

        images_primary.append(frame_data['primary_rgb'])
        images_gripper.append(frame_data['gripper_rgb'])
        images_cat.append(concatenate_rgb_images(frame_data['primary_rgb'], frame_data['gripper_rgb'], vertical=True))
        actions.append(frame_data['rel_actions'])
        if has_gripper_effort:
            gripper_efforts.append(float(frame_data['gripper_max_effort'][0]))

    all_vis = []
    # Use the same sampled frames as the camera and action curves.
    gripper_max_effort_T = (
        np.asarray(gripper_efforts, dtype=np.float32) if has_gripper_effort else None
    )
    actions_vis, fig, ax = plot_action_wrt_time(
        np.array(actions), gripper_max_effort_T=gripper_max_effort_T,
    )
    for frame_idx in range(task_length):
        all_vis.append(concatenate_rgb_images(images_cat[frame_idx], actions_vis[frame_idx],
                                              vertical=False, resize_ratio=1))

    mediapy.show_video(all_vis, fps=fps)

    break

[TCLDataset] total length: 6334
Total tasks: 13
task_idx =  0


100%|█████████████████████████████████████████| 805/805 [00:15<00:00, 52.13it/s]


Plotting action dynamic figures...
